In [12]:
import BECancerResistome
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import math
from sklearn.preprocessing import OrdinalEncoder

# Load Data

## Samples

In [177]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
annotated_control_df = pd.read_csv("data/8_Celular_Context/zscores-unambiguous-VEPannotated-processed-control-with-gene-expression.csv")

/var/folders/ry/v0cp3ptj55qfs_pd6_cghmzm0000gn/T/ipykernel_24323/850768828.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotated_control_df = pd.read_csv("data/8_Celular_Context/zscores-unambiguous-VEPannotated-processed-control-with-gene-expression.csv")


In [4]:
vep_annotated_control_df = pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-control.csv")

/var/folders/ry/v0cp3ptj55qfs_pd6_cghmzm0000gn/T/ipykernel_24323/2333214704.py:1: DtypeWarning: Columns (7,26,111,112,113,114,115) have mixed types. Specify dtype option on import or set low_memory=False.
  vep_annotated_control_df = pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-control.csv")


In [5]:
foldx_sample = pd.read_csv('data/9_ProtVar/Samples/2025.02.10_foldx_energy_sample.csv')
pockets_sample = pd.read_csv('data/9_ProtVar/Samples/2024.05.28_pockets_sample.tsv', sep='\t')
interfaces_sample = pd.read_csv('data/9_ProtVar/Samples/2024.05.28_interface_summary_sample.tsv', sep='\t')

In [187]:
foldx_sample.head()

,uniprot_accession,uniprot_position,alphafold_fragment_id,alphafold_fragment_position,wild_type,mutated_type,foldx_ddg,plddt
0,A0A075B6S0,1,F1,1,N,A,-0.001835,56.9
1,A0A075B6S0,1,F1,1,N,C,-0.011188,56.9
2,A0A075B6S0,1,F1,1,N,D,-0.041779,56.9
3,A0A075B6S0,1,F1,1,N,E,-0.024084,56.9
4,A0A075B6S0,1,F1,1,N,F,0.324027,56.9


In [27]:
pockets_sample.head()

,struct_id,pocket_id,pocket_rad_gyration,pocket_energy_per_vol,pocket_buriedness,pocket_resid,pocket_plddt_mean,pocket_score_combined_scaled
0,A0A024R1R8-F1,1,4.042788,0.316535,0.772959,"{21,22,23,24,25,26,28,29,32}",83.937778,283.034096
1,A0A024R1R8-F1,2,3.175737,0.347111,0.808219,"{12,13,14,15,16,17}",61.206667,102.718057
2,A0A024RBG1-F1,1,7.310256,0.435597,0.856184,"{2,3,4,5,6,7,8,9,10,18,20,21,22,39,40,41,42,47...",89.456190,979.457587
3,A0A024RBG1-F1,2,6.350910,0.389675,0.814896,"{54,57,58,60,61,62,64,65,67,68,69,73,74,75,76,...",83.186923,938.222063
4,A0A024RBG1-F1,3,3.827945,0.378204,0.806045,"{1,2,3,4,5,6,109,110,112,113,114}",77.053636,422.703190


In [28]:
interfaces_sample.head()

,interaction_id,pdockq,uniprot_id1,uniprot_id2,chain1,chain2,ifresid1,ifresid2,sources,n_references,pdb
0,O75106_Q16853,0.74,O75106,Q16853,A,B,"R169,A203,A204,V205,H206,L212,R213,W220,N226,I...","P39,V209,L218,Q219,W226,N232,I233,S234,G235,A2...","BioGRID,humap,intact,string",2,O75106/O75106_Q16853.pdb
1,Q15118_Q15118,0.73,Q15118,Q15118,A,B,"S53,P54,P56,Y179,D182,R183,M186,L255,A257,H304...","S53,P54,P56,Y179,D182,R183,M186,E253,L255,A257...","BioGRID,intact",2,Q15118/Q15118_Q15118.pdb
2,P11142_Q92598,0.73,P11142,Q92598,A,B,"K25,E27,I28,A30,N31,D32,Q33,G34,R36,E48,L50,D5...","R19,A27,N28,E29,F30,S31,R33,N54,T58,Y184,R261,...","BioGRID,corum,humap,intact,otar,string,xlinkdb",9,P11142/P11142_Q92598.pdb
3,Q13326_Q16585,0.73,Q13326,Q16585,A,B,"V40,L41,L43,L44,L47,V48,N50,L51,T54,I55,L58,F6...","V68,I69,L71,L72,L75,A76,I78,N79,I82,I86,M100,F...","corum,otar,string",0,Q13326/Q13326_Q16585.pdb
4,Q13326_Q92629,0.73,Q13326,Q92629,A,B,"K33,L36,Y37,V40,L41,L43,L44,L47,V48,N50,L51,T5...","R30,K31,C33,L34,F37,V38,L40,L41,L44,I45,V47,N4...","corum,string",0,Q13326/Q13326_Q92629.pdb


In [7]:
vep_annotated_control_df.head()

,Guide,Editor,Gene,Cell_Line,Drug,zscore,Hit_class,Source,CRISPR Enzyme,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Strand,Target Transcript ID,RefSeq match transcript (MANE Select),Target Domain,sgRNA Context Sequence,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,hgvs,hgvs ProtVar,Location,Allele,Consequence,IMPACT,Feature_type,Feature,BIOTYPE,EXON,INTRON,HGVSc,HGVSp,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,REF_ALLELE,UPLOADED_ALLELE,DISTANCE,STRAND,FLAGS,SYMBOL_SOURCE,HGNC_ID,CANONICAL,MANE,MANE_SELECT,MANE_PLUS_CLINICAL,TSL,APPRIS,CCDS,ENSP,SWISSPROT,TREMBL,UNIPARC,UNIPROT_ISOFORM,SIFT_pathogenicity,SIFT_class,PolyPhen_pathogenicity,PolyPhen_class,HGVS_OFFSET,AF,gnomADe_AF,gnomADe_AFR_AF,gnomADe_AMR_AF,gnomADe_ASJ_AF,gnomADe_EAS_AF,gnomADe_FIN_AF,gnomADe_MID_AF,gnomADe_NFE_AF,gnomADe_REMAINING_AF,gnomADe_SAS_AF,gnomADg_AF,gnomADg_AFR_AF,gnomADg_AMI_AF,gnomADg_AMR_AF,gnomADg_ASJ_AF,gnomADg_EAS_AF,gnomADg_FIN_AF,gnomADg_MID_AF,gnomADg_NFE_AF,gnomADg_REMAINING_AF,gnomADg_SAS_AF,CLIN_SIG,SOMATIC,PHENO,PUBMED,VAR_SYNONYMS,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS,REVEL,ClinPred,am_class,am_pathogenicity,Enformer_SAD,Enformer_SAR,EVE_CLASS,EVE_SCORE,OpenTargets_geneId,OpenTargets_l2g,Aloft_Confidence,Aloft_Fraction_transcripts_affected,Aloft_pred,Aloft_prob_Dominant,Aloft_prob_Recessive,Aloft_prob_Tolerant,AlphaMissense_pred,AlphaMissense_score,BayesDel_addAF_pred,BayesDel_addAF_score,BayesDel_noAF_pred,BayesDel_noAF_score,DANN_score,DEOGEN2_pred,DEOGEN2_score,ESM1b_pred,ESM1b_score,Eigen-PC-phred_coding,Eigen-PC-raw_coding,Eigen-phred_coding,Eigen-raw_coding,Ensembl_geneid,Ensembl_proteinid,Ensembl_transcriptid,GERP++_NR,GERP++_RS,LIST-S2_pred,LIST-S2_score,MPC_rankscore,MPC_score,MVP_score,MetaLR_pred,MetaLR_score,MetaRNN_pred,MetaRNN_score,MetaSVM_pred,MetaSVM_score,MutFormer_score,MutationAssessor_pred,MutationAssessor_score,MutationTaster_pred,MutationTaster_score,PROVEAN_pred,PROVEAN_score,PrimateAI_pred,Uniprot_acc,VARITY_ER_LOO_score,VARITY_ER_score,VARITY_R_LOO_score,VARITY_R_score,aapos,bStatistic,fathmm-XF_coding_pred,fathmm-XF_coding_score,gMVP_score,genename,phastCons100way_vertebrate,phyloP100way_vertebrate,MaveDB_nt,MaveDB_pro,MaveDB_score,MaveDB_urn,CADD_PHRED,CADD_RAW,MaxEntScan_alt,MaxEntScan_diff,MaxEntScan_ref,SpliceAI_pred_DP_AG,SpliceAI_pred_DP_AL,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,SpliceAI_pred_SYMBOL,BLOSUM62,LOEUF,mutfunc_exp,mutfunc_int,mutfunc_mod,mutfunc_motif,ada_score,rf_score,AA
0,AAAAAACATCGTAGTAGCAG,CBE,RICTOR,A375,PIC,0.274277,non-hit,NaN,SpyoCas9NG,4..8,9606,GRCh38,NC_000005.10,ENSG00000164327,-,ENST00000357387.8,NM_152756.5,CDS,TAACAAAAAACATCGTAGTAGCAGTGATCC,38950510,sense,38950523G>A,C_7,3325C>T,His1109Tyr,Missense,NaN,NaN,ENST00000357387.8:c.3325C>T,NM_152756.5:c.3325C>T,5:38950523-38950523,A,missense_variant,MODERATE,Transcript,ENST00000357387.8,protein_coding,31/38,NaN,ENST00000357387.8:c.3325C>T,ENSP00000349959.3:p.His1109Tyr,3347.0,3325.0,1109.0,H/Y,Cat/Tat,rs1413326068,C,C/T,NaN,-1.0,NaN,HGNC,HGNC:28611,YES,MANE_Select,NM_152756.5,NaN,1.0,P4,CCDS34148.1,ENSP00000349959,Q6R327.172,NaN,UPI00003529F3,Q6R327-1,1.0,tolerated_low_confidence,0.000,benign,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000007,0.000024,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.214,0.463040,likely_benign,0.0685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B,0.0685,T,-0.114447,T,-0.402171,0.970933,T,0.045001,T,-4.388,5.314964,0.516773,4.111072,0.361058,ENSG00000164327,ENSP00000349959,ENST00000357387,5.75,5.75,D,0.917608,0.44121,0.445060,0.093490,T,0.0689,T,0.179100,T,-1.0259,0.000785,N,0.345,".,.,.,.",".,.,.,.",N,-0.22,T,Q6R327,0.157308,0.157308,

In [9]:
vep_annotated_control_df['UNIPROT_ISOFORM'].value_counts()

UNIPROT_ISOFORM
P00533-1    16395
P21359-1     8244
P31749-1     8183
Q02750-1     5358
P21860-1     5340
P04626-1     5088
P01106-2     5077
Q9UGN5-2     4680
Q6R327-1     4638
O00329-1     4404
P26006-2     4272
P10415-1     3152
P21802-1     2688
P29353-6     2490
P04049-1     2202
Q13480-1     2052
P14618-1     1956
P52209-1     1752
P01116-2     1678
P27361-1     1506
P28482-1     1188
P60484-1     1086
P62993-1      858
P01112-1      708
Name: count, dtype: int64

In [19]:
#Create a txt file with all target transcript IDs
transcript_ids = annotated_control_df['Target Transcript ID'].unique()
with open('data/9_ProtVar/Ensembl_transcript_ids.txt', 'w') as f:
    for transcript_id in transcript_ids:
        f.write(f"{transcript_id}\n")

In [14]:
#Open excel file with mapping of transcript IDs to UniProt IDs
transcript_to_uniprot_df = pd.read_excel('data/9_ProtVar/idmapping_2025_10_27.xlsx')

/Users/carolinapinto/Desktop/Tese/BECancerResistome/venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [15]:
transcript_to_uniprot_df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length
0,ENST00000357387.8,Q6R327,reviewed,RICTR_HUMAN,Rapamycin-insensitive companion of mTOR (AVO3 ...,RICTOR KIAA1999,Homo sapiens (Human),1708
1,ENST00000369535.5,P01111,reviewed,RASN_HUMAN,GTPase NRas (EC 3.6.5.2) (Transforming protein...,NRAS HRAS1,Homo sapiens (Human),189
2,ENST00000674063.1,P42338,reviewed,PK3CB_HUMAN,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",PIK3CB PIK3C1,Homo sapiens (Human),1070
3,ENST00000275493.7,P00533,reviewed,EGFR_HUMAN,Epidermal growth factor receptor (EC 2.7.10.1)...,EGFR ERBB ERBB1 HER1,Homo sapiens (Human),1210
4,ENST00000621592.8,P01106,reviewed,MYC_HUMAN,Myc proto-oncogene protein (Class E basic heli...,MYC BHLHE39,Homo sapiens (Human),454
5,ENST00000251849.9,P04049,reviewed,RAF1_HUMAN,RAF proto-oncogene serine/threonine-protein ki...,RAF1 RAF,Homo sapiens (Human),648
6,ENST00000358273.9,P21359,reviewed,NF1_HUMAN,Neurofibromin (Neurofibromatosis-related prote...,NF1,Homo sapiens (Human),2839
7,ENST00000361445.9,P42345,reviewed,MTOR_HUMAN,Serine/threonine-protein kinase mTOR (EC 2.7.1...,MTOR FRAP FRAP1 FRAP2 RAFT1 RAPT1,Homo sapiens (Human),2549
8,ENST00000320031.13,P26006,reviewed,ITA3_HUMAN,Integrin alpha-3 (CD49 antigen-like family mem...,ITGA3 MSK18,Homo sapiens (Human),1051
9,ENST00000448116.7,P29353,reviewed,SHC1_HUMAN,SHC-transforming protein 1 (SHC-transforming p...,SHC1 SHC SHCA,Homo sapiens (Human),583


### Check if protein sequence from the Ensembl Transcript and the UniProt protein match

In [147]:
ensembl_sequence = """MAARRRRSTGGGRARALNESKRVNNGNTAPEDSSPAKKTRRCQRQESKKMPVAGGKANKD
RTEDKQDESVKALLLKGKAPVDPECTAKVGKAHVYCEGNDVYDVMLNQTNLQFNNNKYYL
IQLLEDDAQRNFSVWMRWGRVGKMGQHSLVACSGNLNKAKEIFQKKFLDKTKNNWEDREK
FEKVPGKYDMLQMDYATNTQDEEETKKEESLKSPLKPESQLDLRVQELIKLICNVQAMEE
MMMEMKYNTKKAPLGKLTVAQIKAGYQSLKKIEDCIRAGQHGRALMEACNEFYTRIPHDF
GLRTPPLIRTQKELSEKIQLLEALGDIEIAIKLVKTELQSPEHPLDQHYRNLHCALRPLD
HESYEFKVISQYLQSTHAPTHSDYTMTLLDLFEVEKDGEKEAFREDLHNRMLLWHGSRMS
NWVGILSHGLRIAPPEAPITGYMFGKGIYFADMSSKSANYCFASRLKNTGLLLLSEVALG
QCNELLEANPKAEGLLQGKHSTKGLGKMAPSSAHFVTLNGSTVPLGPASDTGILNPDGYT
LNYNEYIVYNPNQVRMRYLLKVQFNFLQLW"""

ensembl_sequence= ''.join(ensembl_sequence.split())

In [150]:
uniprot_sequence = "MAARRRRSTGGGRARALNESKRVNNGNTAPEDSSPAKKTRRCQRQESKKMPVAGGKANKDRTEDKQDESVKALLLKGKAPVDPECTAKVGKAHVYCEGNDVYDVMLNQTNLQFNNNKYYLIQLLEDDAQRNFSVWMRWGRVGKMGQHSLVACSGNLNKAKEIFQKKFLDKTKNNWEDREKFEKVPGKYDMLQMDYATNTQDEEETKKEESLKSPLKPESQLDLRVQELIKLICNVQAMEEMMMEMKYNTKKAPLGKLTVAQIKAGYQSLKKIEDCIRAGQHGRALMEACNEFYTRIPHDFGLRTPPLIRTQKELSEKIQLLEALGDIEIAIKLVKTELQSPEHPLDQHYRNLHCALRPLDHESYEFKVISQYLQSTHAPTHSDYTMTLLDLFEVEKDGEKEAFREDLHNRMLLWHGSRMSNWVGILSHGLRIAPPEAPITGYMFGKGIYFADMSSKSANYCFASRLKNTGLLLLSEVALGQCNELLEANPKAEGLLQGKHSTKGLGKMAPSSAHFVTLNGSTVPLGPASDTGILNPDGYTLNYNEYIVYNPNQVRMRYLLKVQFNFLQLW"

In [151]:
ensembl_sequence == uniprot_sequence

True

In [184]:
transcript_mapping= pd.read_csv("data/transcriptID_mapping.csv")

In [175]:
#drop columns unnamed 5 and unnamed 6
transcript_mapping = transcript_mapping.drop(columns=['Unnamed: 5', 'Unnamed: 6'])


In [180]:
transcript_mapping.to_csv("data/transcriptID_mapping.csv", sep=',', index=False)

In [185]:
transcript_mapping

,GeneID,TranscriptID_Ensembl_isoform,UniProtID_isoform,is_canonical_in_uniprot,Protein_Sequence
0,EGFR,ENST00000275493.7,P00533-1,True,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
1,KRAS,ENST00000311936.8,P01116-2,False,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
2,NRAS,ENST00000369535.5,P01111,True,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
3,HRAS,ENST00000311189.8,P01112-1,True,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
4,BRAF,ENST00000646891.2,P15056,True,MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEE...
5,RAF1,ENST00000251849.9,P04049-1,True,MEHIQGAWKTISNGFGFKDAVFDGSSCISPTIVQQFGYQRRASDDG...
6,ERBB3,ENST00000267101.8,P21860-1,True,MRANDALQVLGLLFSLARGSEVGNSQAVCPGTLNGLSVTGDAENQY...
7,ERBB2,ENST00000269571.10,P04626-1,True,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...
8,MAPK1,ENST00000215832.11,P28482-1,True,MAAAAAAGAGPEMVRGQVFDVGPRYTNLSYIGEGAYGMVCSAYDNV...
9,MAPK3,ENST00000263025.9,P27361-1,True,MAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQ...


In [ ]:
MC_EG_proteins_uniprot={
    'P00533',
    'P01116',
    'P01111',
    'P01112',
    'P15056',
    'P04049',
    'P21860',
    'P04626',
    'P28482',
    'P27361',
    'Q02750',
    'P36507',
    'P42336',
    'P42338',
    'O00329',
    'P31749',
    'P01106',
    'P29353',
    'P62993',
    'Q13480',
    'P26006',
    'P08069',
    'P21802',
    'P42345',
    'Q6R327',
    'P21359',
    'P60484',
    'P14618',
    'Q53GA4',
    'P52209',
    'P10415',
    'P09874',
    'Q9UGN5'
}

# Predicted Changes in Stability

In [ ]:
#run on server
foldx = pd.read_csv('/data/benchmarks/genomic_annot/afdb_foldx_export_20250210.csv') 

In [ ]:
foldx.shape

In [ ]:
foldx.head()

In [ ]:
#Check if uniprot_acession_proteins are in foldx dataset - 'True' if they are
set(MC_EG_proteins_uniprot).issubset(set(foldx['uniprot_accession'].unique()))

## Merge foldx_ddg with the annotated dataset

In [ ]:
#Load the transcript mapping ID dataset
transcriptID_mapping = pd.read_csv('/data/benchmarks/genomic_annot/transcriptID_mapping.csv', sep=';')

In [ ]:
transcriptID_mapping

In [ ]:
#Map Ensembl Transcript to uniprot accession
mapping_dict = dict(zip(transcriptID_mapping["TranscriptID_Ensembl_isoform"], transcriptID_mapping["UniProtID_global"]))
annotated_control_df["uniprot_accession"] = annotated_control_df["Target Transcript ID"].map(mapping_dict)

In [ ]:
# Change column position
cols = list(annotated_control_df.columns)

idx = cols.index("Target Transcript ID")
cols.remove("uniprot_accession")
cols.insert(idx + 1, "uniprot_accession")

annotated_control_df = annotated_control_df[cols]

In [ ]:
annotated_control_df.head(20)

#### Get wild type, mutated type and AA position from 'Amino Acid Edits'

In [ ]:
#Apply convert_variant to generate "H1109Y"-like string

# Create a mask for non-NC entries
mask = annotated_control_df["Amino Acid Edits"] != "(NC)"

# Apply conversion function only to valid entries
annotated_control_df.loc[mask, "aa_change"] = annotated_control_df.loc[mask, "Amino Acid Edits"].map(
    BECancerResistome.convert_amino_acid_variants
)

# Fill the others with "(NC)" if needed
annotated_control_df.loc[~mask, "aa_change"] = "(NC)"

In [ ]:
# Change column position
cols = list(annotated_control_df.columns)

idx = cols.index("Amino Acid Edits")
cols.remove("aa_change")
cols.insert(idx + 1, "aa_change")

annotated_control_df = annotated_control_df[cols]

In [ ]:
#Split aa_change into wild_type, position, mutated_type (with NC masking)

annotated_control_df.loc[mask, "wild_type"] = annotated_control_df.loc[mask, "aa_change"].str[0]
annotated_control_df.loc[mask, "uniprot_position"] = annotated_control_df.loc[mask, "aa_change"].str[1:-1].astype(int)
annotated_control_df.loc[mask, "mutated_type"] = annotated_control_df.loc[mask, "aa_change"].str[-1]

# Fill rest with NaN
annotated_control_df.loc[~mask, ["wild_type", "uniprot_position", "mutated_type"]] = np.nan

#### Merge foldx_ddg 

In [ ]:
annotated_foldx_control_df = annotated_control_df.merge(
    foldx[["uniprot_accession", "uniprot_position", "wild_type", "mutated_type", "foldx_ddg", "plddt"]],
    how="left",
    on=["uniprot_accession", "uniprot_position", "wild_type", "mutated_type"]
)

In [ ]:
annotated_foldx_control_df.head(20)

In [ ]:
#Filter rows with AA changes for missingness analysis 
annotated_foldx_control_df_aa = annotated_foldx_control_df[annotated_foldx_control_df["aa_change"] != "(NC)"]

In [ ]:
missing_values = annotated_foldx_control_df['foldx_ddg'].isna().sum()
percent_missing = 100 * missing_values / len(annotated_foldx_control_df)

print(f'Missing foldx ddg values in the dataset: {missing_values}')
print(f'Missing foldx ddg values in the dataset (%): {percent_missing}')

In [ ]:
#Missing values that are not because of the absence of aa change
missing_values_aa = annotated_foldx_control_df_aa['foldx_ddg'].isna().sum()
percent_missing_aa = 100 * missing_values_aa / len(annotated_foldx_control_df_aa)

print(f'Missing foldx ddg values in the dataset: {missing_values_aa}')
print(f'Missing foldx ddg values in the dataset (%): {percent_missing_aa}')

In [ ]:
annotated_foldx_control_df.to_csv("/data/benchmarks/genomic_annot/zscores-unambiguous-VEPannotated-processed-gene-expression-foldx-control")

In [ ]:
# Filter rows where foldx_ddg is missing
missing_foldx = annotated_foldx_control_df_aa[annotated_foldx_control_df_aa["foldx_ddg"].isna()]

# Get distribution of 'Hit_class' in those rows
hit_class_distribution = missing_foldx["Hit_class"].value_counts(dropna=False)

# Show both counts and percentages
hit_class_percent = missing_foldx["Hit_class"].value_counts(normalize=True, dropna=False) * 100

# Display nicely
distribution_df = pd.DataFrame({
    "Count": hit_class_distribution,
    "Percentage": hit_class_percent.round(2)
})

display(distribution_df)

In [ ]:
len(missing_foldx)

In [ ]:
# Count missing foldx_ddg values grouped by uniprot_accession
missing_by_uniprot = (
    annotated_foldx_control_df_aa[annotated_foldx_control_df_aa["foldx_ddg"].isna()]
    .groupby("uniprot_accession")
    .size()
    .reset_index(name="Missing_count")
    .sort_values(by="Missing_count", ascending=False)
)

display(missing_by_uniprot)